In [ ]:
# Lecture 18 — UDFs: built-in vs UDF, and the plan difference
# Spark 101 · Module 5 · Lecture 18
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
# --- A small DataFrame to transform ---------------------------------------
df = spark.createDataFrame(
    [(1, "ada"), (2, "linus"), (3, "grace")],
    ["id", "name"],
)
df.show()

In [ ]:
# --- The built-in way: upper(). Catalyst-friendly, runs in the JVM. --------
from pyspark.sql.functions import col, upper

builtin = df.withColumn("name_upper", upper(col("name")))
builtin.show()

In [ ]:
# --- The UDF way: your own Python function, wrapped, applied per row -------
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def to_upper(s):
    # arbitrary Python — Catalyst cannot see inside this
    return s.upper() if s is not None else None

to_upper_udf = udf(to_upper, StringType())   # declare the RETURN type

via_udf = df.withColumn("name_upper", to_upper_udf(col("name")))
via_udf.show()
# Same result on screen — very different machinery underneath.

In [ ]:
# --- The proof: look at the plans -----------------------------------------
print("=== built-in plan ===")
builtin.explain()
# `upper` appears as a native expression Catalyst understands.

print("=== UDF plan ===")
via_udf.explain()
# Look for `BatchEvalPython` — that node is Spark shipping every row out to a
# Python process and back. That round trip is the cost, once per row.

In [ ]:
# --- Before writing a UDF, search the built-ins ----------------------------
# Common "I need a UDF" tasks that already have a built-in:
from pyspark.sql.functions import concat_ws, coalesce, when, lit, round as _round

df.select(
    concat_ws(" ", lit("user:"), col("name")).alias("labelled"),   # string work
    when(col("id") > 1, "repeat").otherwise("first").alias("kind"), # branching
    coalesce(col("name"), lit("unknown")).alias("safe_name"),       # null handling
).show()

In [ ]:
# A UDF with TWO inputs.
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType
@udf(StringType())
def greet(name, n):
    return f"Hello {name}" + ("!" * int(n))
df.withColumn("greeting", greet(col("name"), col("id"))).show(truncate=False)

In [ ]:
# Register a UDF so it works in SQL too.
spark.udf.register("to_upper_sql", to_upper, StringType())
df.createOrReplaceTempView("people")
spark.sql("SELECT id, to_upper_sql(name) AS up FROM people").show()

In [ ]:
# Vectorized pandas UDF — processes a batch at a time, far cheaper on big data.
from pyspark.sql.functions import pandas_udf
@pandas_udf(StringType())
def pud_upper(s):            # s is a pandas Series
    return s.str.upper()
df.withColumn("u", pud_upper(col("name"))).show()

In [ ]:
# The built-in is still first choice — same result, fully optimized.
from pyspark.sql.functions import upper
df.withColumn("u", upper(col("name"))).explain()

In [ ]:
# Recap:
#   - A UDF is a Python function wrapped with a declared return type.
#   - It is OPAQUE to Catalyst and pays JVM<->Python serialization per row.
#   - Search pyspark.sql.functions FIRST; reach for a UDF only when nothing fits.
#   - On hot data, a vectorized pandas UDF is far faster than a plain UDF.
# End of Module 5 — DataFrames & Spark SQL.